In [13]:
# Import required libraries
import pandas as pd
import numpy as np
import os

In [1]:
%env KAGGLE_API_TOKEN=KGAT_ba1b30339292569a0f8e372c0bdadde7
%env KAGGLE_KEY=KGAT_ba1b30339292569a0f8e372c0bdadde7
%env KAGGLE_USERNAME=brianhill1398

env: KAGGLE_API_TOKEN=KGAT_ba1b30339292569a0f8e372c0bdadde7
env: KAGGLE_KEY=KGAT_ba1b30339292569a0f8e372c0bdadde7
env: KAGGLE_USERNAME=brianhill1398


In [3]:
ENV = "google_colab"
!kaggle competitions download -c cmi-flu-internal-prediction-challenge
!unzip cmi-flu-internal-prediction-challenge.zip
DATA_PATH = os.path.join(os.getcwd(),'PART2-26-02-13_reorg') if ENV == 'google_colab' else 'data/PART2-26-02-13_reorg'

100% 4.20G/4.20G [00:46<00:00, 97.5MB/s]

Archive:  cmi-flu-internal-prediction-challenge.zip
  inflating: PART2-26-01-26/PART2-26-01-26/challenge_events.tsv  
  inflating: PART2-26-01-26/PART2-26-01-26/challenge_flow.tsv  
  inflating: PART2-26-01-26/PART2-26-01-26/challenge_hai.tsv  
  inflating: PART2-26-01-26/PART2-26-01-26/challenge_participants.tsv  
  inflating: PART2-26-01-26/PART2-26-01-26/challenge_transcriptomics_2024_UGA.tsv  
  inflating: PART2-26-01-26/PART2-26-01-26/columns.tsv  
  inflating: PART2-26-01-26/PART2-26-01-26/md5sum.txt  
  inflating: PART2-26-01-26/PART2-26-01-26/train_array.tsv  
  inflating: PART2-26-01-26/PART2-26-01-26/train_bcr.tsv  
  inflating: PART2-26-01-26/PART2-26-01-26/train_events.tsv  
  inflating: PART2-26-01-26/PART2-26-01-26/train_flow.tsv  
  inflating: PART2-26-01-26/PART2-26-01-26/train_hai.tsv  
  inflating: PART2-26-01-26/PART2-26-01-26/train_participants.tsv  
  inflating: PART2-26-01-26/PART2-26-01-26/train_transcriptomics_2019_UGA.t

In [16]:
# Load the dataset
DATA_PATH = os.path.join(os.getcwd(),'PART2-26-02-13_reorg') if ENV == 'google_colab' else 'data/PART2-26-02-13_reorg'
df = pd.read_csv(DATA_PATH + "/train_hai.tsv", sep='\t')
# Load the dataset
#df = pd.read_csv('../data/train_hai.tsv', sep='\t')
#df = df.drop(columns=['hai_id'])
df = df.drop(columns=['hai_id', 'material'])
df['value'] = pd.to_numeric(df['value'], errors='coerce')
df = df[df['timepoint'].isin([0.0, 28.0, 365.0])]
df.head()

,participant_id,timepoint,virus_strain,value
0,2016_UGA.ID_001,0.0,H1N1 A/South Carolina/1/1918,20.0
1,2016_UGA.ID_001,28.0,H1N1 A/South Carolina/1/1918,40.0
2,2016_UGA.ID_002,0.0,H1N1 A/South Carolina/1/1918,5.0
3,2016_UGA.ID_002,28.0,H1N1 A/South Carolina/1/1918,5.0
4,2016_UGA.ID_003,0.0,H1N1 A/South Carolina/1/1918,80.0


In [18]:
df['hai_timepoint'] = 'HAI_' + df['virus_strain'].astype(str) + '_' + df['timepoint'].astype(int).astype(str)

# Pivot the table
df_pivot = df.pivot_table(
    index='participant_id',
    columns='hai_timepoint',
    values='value'
)

# Reset index to make participant_id a regular column
df_pivot = df_pivot.reset_index()
df_pivot = df_pivot.rename_axis(None, axis=1)
df_pivot.drop(columns=["HAI_-_0","HAI_-_28"], inplace = True)

for col in df_pivot.columns:
    if col != 'participant_id':
        df_pivot[col] = df_pivot[col].apply(lambda x: np.log(x) if pd.notna(x) else x)

df_pivot

,participant_id,HAI_Anc B/Lee/1940_0,HAI_Anc B/Lee/1940_28,HAI_Anc B/Lee/1940_365,HAI_Anc B/Maryland/1959_0,HAI_Anc B/Maryland/1959_28,HAI_Anc B/Singapore/1964_0,HAI_Anc B/Singapore/1964_28,HAI_H1N1 A/Beijing/262/1995_0,HAI_H1N1 A/Beijing/262/1995_28,...,HAI_Yam B/Sichuan/379/1999_365,HAI_Yam B/Texas/6/2011_0,HAI_Yam B/Texas/6/2011_28,HAI_Yam B/Texas/6/2011_365,HAI_Yam B/Wisconsin/1/2010_0,HAI_Yam B/Wisconsin/1/2010_28,HAI_Yam B/Wisconsin/1/2010_365,HAI_Yam B/Yamagata/16/1988_0,HAI_Yam B/Yamagata/16/1988_28,HAI_Yam B/Yamagata/16/1988_365
0,2016_UGA.ID_001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.688879,3.688879,...,NaN,5.075174,5.768321,NaN,5.768321,5.768321,NaN,5.075174,5.768321,NaN
1,2016_UGA.ID_002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.382027,3.688879,...,NaN,3.688879,4.382027,NaN,4.382027,4.382027,NaN,3.688879,3.688879,NaN
2,2016_UGA.ID_003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.382027,3.688879,...,NaN,5.768321,5.768321,NaN,5.768321,5.768321,NaN,5.768321,5.768321,NaN
3,2016_UGA.ID_004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.302585,3.688879,...,NaN,2.995732,4.382027,NaN,3.688879,4.382027,NaN,2.302585,3.688879,NaN
4,2016_UGA.ID_005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.302585,1.609438,...,4.382027,4.382027,4.382027,3.688879,4.382027,4.382027,3.688879,3.688879,3.688879,3.688879
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3752,SDY887.SUB134259,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3753,SDY887.SUB134260,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3754,SDY887.SUB197783,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3755,SDY887.SUB197784,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df.to_csv('../cleaned_data/hai_cleaned.csv', index=False)